In [1]:
# =========================
# CELL 1 — INSTALL
# =========================

# Run this once

!pip install groq google-generativeai gradio pandas openpyxl

In [2]:
# =========================
# CELL 2 — IMPORTS
# =========================

import json
import sys
import pandas as pd
import gradio as gr
from groq import Groq
import google.generativeai as genai
from google.colab import userdata
from IPython.display import display

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# =========================
# CELL 3 — API KEYS
# =========================

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

In [4]:
# =========================
# CELL 4 — CLIENTS
# =========================

groq_client = Groq(
    api_key=GROQ_API_KEY
)

genai.configure(
    api_key=GEMINI_API_KEY
)

In [5]:
# =========================
# CELL 5 — MODELS
# =========================

MODELS = {

    # =====================
    # GROQ MODELS
    # =====================

    "llama": {
        "provider": "groq",
        "model": "llama-3.3-70b-versatile"
    },

    # =====================
    # GEMINI MODELS
    # =====================

    "gemini-flash": {
        "provider": "gemini",
        "model": "gemini-2.5-flash"
    },

    "gemini-pro": {
        "provider": "gemini",
        "model": "gemini-2.5-pro"
    }
}

In [6]:
# =========================
# CELL 6 — DATASET GENERATOR
# =========================

def generate_dataset(topic, rows, model_name):

    prompt = f"""
    You are a dataset generator.

    Generate {rows} rows of synthetic data about:
    {topic}

    Rules:
    - Return ONLY valid JSON
    - No markdown
    - No explanation
    - Output must start with [
    """

    provider = MODELS[model_name]["provider"]

    selected_model = MODELS[model_name]["model"]

    try:

        # =====================
        # GROQ
        # =====================

        if provider == "groq":

            response = groq_client.chat.completions.create(
                model=selected_model,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=1
            )

            content = response.choices[0].message.content

        # =====================
        # GEMINI
        # =====================

        elif provider == "gemini":

            model = genai.GenerativeModel(
                selected_model
            )

            response = model.generate_content(
                prompt
            )

            content = response.text

        # =====================
        # CLEAN JSON
        # =====================

        content = content.replace(
            "```json",
            ""
        )

        content = content.replace(
            "```",
            ""
        )

        content = content.strip()

        # =====================
        # LOAD JSON
        # =====================

        data = json.loads(content)

        df = pd.DataFrame(data)

        # =====================
        # NOTEBOOK OUTPUT
        # =====================

        print("\n========== GENERATED DATASET ==========\n")

        print(f"MODEL USED: {model_name}")

        print(f"\nTOTAL ROWS: {len(df)}")

        print("\nCOLUMNS:\n")

        print(df.columns.tolist())

        print("\nFIRST 5 ROWS:\n")

        display(df.head())

        print("\n=======================================\n")

        sys.stdout.flush()

        # =====================
        # SAVE FILES
        # =====================

        csv_file = "dataset.csv"

        json_file = "dataset.json"

        excel_file = "dataset.xlsx"

        df.to_csv(
            csv_file,
            index=False
        )

        df.to_json(
            json_file,
            orient="records",
            indent=2
        )

        df.to_excel(
            excel_file,
            index=False
        )

        return (
            df,
            csv_file,
            json_file,
            excel_file
        )

    except Exception as e:

        print("\n========== ERROR ==========\n")

        print(str(e))

        if "content" in locals():

            print("\nRAW OUTPUT:\n")

            print(content)

        print("\n===========================\n")

        sys.stdout.flush()

        error_df = pd.DataFrame({
            "Error": [str(e)]
        })

        return (
            error_df,
            None,
            None,
            None
        )

In [9]:
# =========================
# CELL 7 — GRADIO UI
# =========================

demo = gr.Interface(

    fn=generate_dataset,

    inputs=[

        gr.Textbox(
            label="Dataset Topic",
            placeholder="Example: Hospital Patients"
        ),

        gr.Slider(
            1,
            50,
            value=10,
            step=1,
            label="Rows"
        ),

        gr.Dropdown(
            choices=[
                "llama",
                "gemini-flash",
                "gemini-pro"
            ],

            value="llama",

            label="Model"
        )
    ],

    outputs=[

        gr.Dataframe(
            label="Generated Dataset"
        ),

        gr.File(
            label="Download CSV"
        ),

        gr.File(
            label="Download JSON"
        ),

        gr.File(
            label="Download Excel"
        )
    ],

    title="AI Synthetic Dataset Generator",

    description="""
    Generate synthetic datasets using:

    • Groq Llama
    • Gemini Flash
    • Gemini Pro
    """,

    allow_flagging="never"
)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


In [10]:
# Stop previous server if running
demo.close()

# Launch Gradio
demo.launch(
    debug=True,
    share=True,
    inline=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0c5d557619516a07e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



========== GENERATED DATASET ==========

MODEL USED: llama

TOTAL ROWS: 40

COLUMNS:

['PatientID', 'Name', 'Age', 'Gender', 'Department', 'AdmitDate']

FIRST 5 ROWS:



,PatientID,Name,Age,Gender,Department,AdmitDate
0,1,John Doe,35,Male,Cardiology,2022-01-01
1,2,Jane Smith,42,Female,Neurology,2022-01-05
2,3,Michael Brown,28,Male,Oncology,2022-01-10
3,4,Emily Davis,38,Female,Pediatrics,2022-01-15
4,5,David Lee,45,Male,Orthopedics,2022-01-20




Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0c5d557619516a07e4.gradio.live
